# Pods
## List Running Pods

``` sh
kubectl get pods -A
```

Kubernetes lets you divide your cluster into namespaces. Each namespace can have
its own set of resources. The above command lists all running pods on every
cluster. Pods in the kube-system namespace belong to Kubernetes and helps it
function.


## “SSH”

To connect to the application look at the namespaces:

``` sh
kubectl get pods --all-namespaces
kubectl exec -it -n <namespace> <pod> -c <container> -- bash
```


## Logs

``` sh
kubectl get pods --all-namespaces
kubectl logs -f -n <namespace> <pod> -c <container>
```

This lets you view the logs of the running pod. The container running on the pod
should be configured to output logs to STDOUT/STDERR.

## Describe Pods

Troubleshooting Pods

kubectl describe pods

Common Errors:

 - OOMError
 - CrashLoopBackup
 - ImageNotFound

## Restart a Pod

If you pod is not responding or needs a restart the way to do it is to use the
following command. This will delete the pod and replace it with a new pod if it
is a part of a deployment.

kubectl delete pod <pod-name>

## Deployments

### Scale Down / Up

This has to be done through the deployment in the helm chart. Another way to do it is to scale down

```
num=0

kubectl scale --replicas=${num} -n <namespace> deployment/<deploymentname>
```

## Nodes

Restarting nodes may need to happen if you need to change the size of the
instance, the machine's disk gets full, or you need to update a new AMI.  The
following code provides the howto.

### EKS


```
export AWS_PROFILE=aws_profile
export KUBECONFIG=./kubeconfig

for i in $(kubectl get nodes | awk '{print $1}' | grep -v NAME)
do
        kubectl drain --ignore-daemonsets --grace-period=60 --timeout=30s --force $i
        aws ec2 terminate-instances --instance-ids $(aws ec2 describe-instances --filter "Name=private-dns-name,Values=${i}" | jq -r '.Reservations[].Instances[].InstanceId')
        sleep 300 # Wait 5 mins for the new machine to come back up
done
```
